# 02 — Blockchain Blocks Extraction
## Extraction complète des métriques de blocs depuis genesis (bloc 0)

**Prérequis** : Bitcoin Core synchronisé avec `txindex=1`
**Durée estimée** : 24-48h pour ~890 000 blocs
**Volume GCS** : ~5 GB Parquet

**Métriques extraites par bloc** :
- Volume et activité (tx count, fees, coinbase reward)
- Distribution complète des fee rates (p10→p99)
- Distribution des montants (whale vs retail)
- SegWit, Taproot, RBF adoption
- UTXO création/destruction
- Timing inter-blocs, halving epoch, difficulté

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Imports et configuration
# ════════════════════════════════════════════════════════════════════════
import os, sys, time, json
sys.path.insert(0, "/workspace")

import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display
import ipywidgets as widgets

from btc_pipeline.storage.gcs_client import StorageClient
from btc_pipeline.config import Config
from btc_pipeline.collectors.bitcoin_core import (
    get_rpc, get_chain_info, run_blocks_extraction,
)

os.environ.setdefault("GCS_BUCKET_NAME", "btc-training-data")
os.environ.setdefault("STORAGE_BACKEND", "gcs")
os.environ.setdefault("WORKSPACE_DIR",  "/workspace")

storage = StorageClient()
config = Config()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Vérifications
# ════════════════════════════════════════════════════════════════════════
import shutil
_, _, free = shutil.disk_usage(os.environ.get("WORKSPACE_DIR", "/workspace"))
print(f"✅ Espace libre : {free/1e9:.1f} GB")

# Vérifier Bitcoin Core
try:
    rpc = get_rpc()
    info = get_chain_info(rpc)
    print(f"✅ Bitcoin Core: {info['blocks']:,} blocs, synced={'✅' if info['blocks']==info['headers'] else '⏳'}")
except Exception as e:
    print(f"❌ Bitcoin Core non disponible: {e}")
    print("   → Utiliser le notebook alternatif avec Blockchair")
    raise SystemExit("Bitcoin Core requis pour ce notebook")

state = storage.get_pipeline_state()
last_height = state.get("tasks", {}).get("blockchain_blocks", {}).get("last_completed_height", -1)
remaining = info["blocks"] - last_height
est_hours = remaining * 0.15 / 3600
print(f"📊 Blocs restants : {remaining:,} (depuis hauteur {last_height + 1})")
print(f"⏱️  Estimation : ~{est_hours:.1f}h")

## Extraction des blocs
Lance l'extraction complète. Reprend automatiquement depuis le dernier bloc traité.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Extraction des blocs
# ════════════════════════════════════════════════════════════════════════
progress = widgets.IntProgress(
    value=0, max=remaining,
    description="Blocs:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)
status_label = widgets.HTML(value="Démarrage...")
display(widgets.VBox([progress, status_label]))

def update_progress(current, total, info):
    progress.value = min(current, progress.max)
    status_label.value = (
        f"Bloc <b>{info.get('height', '?'):,}</b> — "
        f"{info.get('rate', 0):.0f} blocs/s — "
        f"ETA: {info.get('eta_h', 0):.1f}h"
    )

stats = run_blocks_extraction(
    storage,
    batch_size=5000,
    config=config,
    progress_callback=update_progress,
)

print(f"\n✅ Extraction terminée")
print(f"   Blocs: {stats['total_rows']:,}")
print(f"   Volume: {stats['total_gb']:.2f} GB")
print(f"   Durée: {stats['duration_s']/3600:.1f}h")
print(f"   Batches: {stats['batches']}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Validation
# ════════════════════════════════════════════════════════════════════════
# Charger un batch récent pour vérification
files = storage.list_files("raw/blockchain/blocks/")
if files:
    last_file = sorted(files)[-1]
    df_sample = storage.download_parquet(last_file)
    print(f"Dernier batch : {last_file}")
    print(f"  Lignes     : {len(df_sample):,}")
    print(f"  Colonnes   : {len(df_sample.columns)}")
    print(f"  Hauteurs   : {df_sample['block_height'].min():,} → {df_sample['block_height'].max():,}")
    print(f"  Timestamps : {pd.to_datetime(df_sample['block_timestamp'].min(), unit='s')} → "
          f"{pd.to_datetime(df_sample['block_timestamp'].max(), unit='s')}")
    print(f"\nStatistiques clés :")
    print(df_sample[["tx_count", "total_fees_btc", "fee_rate_p50_sat_vbyte",
                      "segwit_tx_ratio", "block_size_bytes"]].describe().round(2))
else:
    print("⚠️  Aucun fichier de blocs trouvé sur GCS")